# Project: Predict Medical Insurance Costs with Linear Regression

**Goal:** predict how much a person is charged for health insurance (`charges`) from their age, BMI, number of children, sex, smoking status and region.

**Dataset:** the classic `insurance.csv` (1,338 people, 7 columns). It loads directly from the internet, so there is nothing to upload.

**Steps:** 1) Load data, 2) Explore, 3) Visualize, 4) Prepare, 5) Train, 6) Evaluate, 7) Understand the model, 8) Predict for a new person, 9) Improve the model.

Run the cells from top to bottom (Shift + Enter).

## Step 1: Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
df = pd.read_csv(url)
df.head()

## Step 2: Explore the data

Check the size, missing values and basic statistics. Every row is one person. `charges` is the **label** we want to predict; the other six columns are the **features**.

In [ ]:
print('Rows, columns:', df.shape)
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
df.describe()

## Step 3: Visualize

Look for patterns before modelling. Notice how strongly **smoker** separates the charges.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

sns.histplot(df['charges'], kde=True, ax=axes[0])
axes[0].set_title('Distribution of charges')

sns.boxplot(x='smoker', y='charges', data=df, ax=axes[1])
axes[1].set_title('Charges: smokers vs non-smokers')

sns.scatterplot(x='age', y='charges', hue='smoker', data=df, ax=axes[2])
axes[2].set_title('Age vs charges')

plt.tight_layout()
plt.show()

## Step 4: Prepare the data

Linear regression only understands numbers, so text columns (`sex`, `smoker`, `region`) are converted into 0/1 columns. This is called **one-hot encoding**. `drop_first=True` removes one redundant column per category.

Then we split into **80% training** and **20% test** data. The test data is kept hidden from the model to check it honestly.

In [ ]:
from sklearn.model_selection import train_test_split

df_encoded = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True, dtype=int)
print(df_encoded.columns.tolist())

X = df_encoded.drop('charges', axis=1)   # features
y = df_encoded['charges']                # label

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Training rows:', len(X_train), '| Test rows:', len(X_test))

## Step 5: Train the model

`fit` finds the best weight for every feature plus a bias (intercept), minimizing the mean squared error on the training data.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
print('Training finished.')

## Step 6: Evaluate on unseen test data

- **MAE:** average error in rupee/dollar terms.
- **RMSE:** like MAE but punishes big errors more.
- **R²:** 1.0 is perfect, 0 is no better than always guessing the average.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pred = model.predict(X_test)

print('MAE  :', round(mean_absolute_error(y_test, pred), 2))
print('RMSE :', round(np.sqrt(mean_squared_error(y_test, pred)), 2))
print('R2 (test) :', round(r2_score(y_test, pred), 4))
print('R2 (train):', round(model.score(X_train, y_train), 4))

plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.6)
lims = [y.min(), y.max()]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual charges')
plt.ylabel('Predicted charges')
plt.title('Actual vs predicted')
plt.legend()
plt.show()

**How to read this:** the closer the dots are to the red dashed line, the better the model. If train R² and test R² are close, the model is not overfitting. You should see R² on the test set of roughly 0.75 to 0.80 (it can vary slightly with the library version).

## Step 7: Understand what the model learned

Each coefficient says: *if this feature goes up by 1 (others unchanged), the predicted charge changes by this amount.*

In [ ]:
coef = pd.DataFrame({'feature': X.columns, 'coefficient': model.coef_})
coef = coef.sort_values('coefficient', ascending=False)
print('Intercept (bias):', round(model.intercept_, 2))
print()
print(coef.to_string(index=False))

plt.figure(figsize=(8, 4))
sns.barplot(x='coefficient', y='feature', data=coef)
plt.title('Effect of each feature on charges')
plt.show()

You should see that **`smoker_yes` has by far the largest effect**: smoking adds roughly 23,000 to the predicted charge, while each extra year of age adds roughly 250.

## Step 8: Predict for a brand-new person

Change the values and re-run to see how the prediction moves.

In [ ]:
person = {
    'age': 30, 'bmi': 27.0, 'children': 1,
    'sex_male': 1, 'smoker_yes': 0,
    'region_northwest': 0, 'region_southeast': 1, 'region_southwest': 0,
}
new_X = pd.DataFrame([person])[X.columns]
print('Non-smoker:', round(model.predict(new_X)[0], 2))

person['smoker_yes'] = 1
new_X = pd.DataFrame([person])[X.columns]
print('Same person as smoker:', round(model.predict(new_X)[0], 2))

## Step 9: Improve the model

The scatter plot in Step 3 showed that the relationship is not perfectly straight, and that smoking combined with high BMI is especially costly. We can help a *linear* model capture this by adding two engineered features: `age_sq` (a curve in age) and `smoker_bmi` (smoker × BMI interaction).

In [ ]:
X2 = X.copy()
X2['age_sq'] = X2['age'] ** 2
X2['smoker_bmi'] = X2['smoker_yes'] * X2['bmi']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42
)

model2 = LinearRegression().fit(X2_train, y2_train)
pred2 = model2.predict(X2_test)

print('Original model  R2:', round(r2_score(y_test, pred), 4), '| MAE:', round(mean_absolute_error(y_test, pred), 2))
print('Improved model  R2:', round(r2_score(y2_test, pred2), 4), '| MAE:', round(mean_absolute_error(y2_test, pred2), 2))

## Practice ideas

1. Change `random_state` or `test_size` in Step 4 and see how R² moves.
2. Train a model using only `age` (simple linear regression) and compare its R² with the full model.
3. Try `np.log(y)` as the label to reduce the effect of very large charges.
4. Try `RandomForestRegressor` from `sklearn.ensemble` and compare it with linear regression.